# TP2c — <!-- BEGIN SOLUTION -->SOLUTION (instructors only) — <!-- END SOLUTION -->Climatology vs interannual variability: 31 years of SST

**Suggested duration: ~1h30** · **Prerequisites**: TP2a (Copernicus Marine login, the `download()` helper), Course 1.

In TP1b and TP2a you worked on **one year**, 2023. Was it a normal year? To answer, you need the **typical year**, the **climatology**, computed over many years. Then the **anomaly** tells you what was unusual:

$$\text{anomaly} = \text{observation} - \text{climatology}$$

In this notebook you download **31 years** (1993–2023) of monthly sea surface temperature from the GLORYS12 ocean reanalysis, and you find El Niño and La Niña in it.

## Learning objectives
By the end of this notebook you should be able to:
1. Download a **long time series** efficiently (monthly, one depth level).
2. Compute an **area-weighted** mean over a box.
3. Compute a **monthly climatology** with `groupby("time.month")`.
4. Compute **anomalies**, and recognise **El Niño** and **La Niña** events.
5. **Cross-check** a climatology with an independent method.

## How to work through this notebook
- Cells marked `# TODO` contain code **you need to complete**. Everything else is already written.
- After each exercise, a **✅ Check** box tells you what a correct result should look like.
- **🚀 Advanced track** blocks are optional: do them only if you finished the main exercise.
- Stuck on a `# TODO`? Below each exercise, a **💡 Hint** box (and, as a last resort, a **🔑 Solution** box) can be opened with a click.
- Stuck for more than 10 minutes? Ask a trainer — that's what we're here for.

> **⚠️ Before you start: work on your own copy.** In the JupyterLab file browser, right-click this notebook → *Duplicate*, then rename the copy to `02c_interannual_EN_mine.ipynb` and open it.

---
## Setup — the same as TP2a

Run this cell as is: the same study box, `download()`, `show_dataset()` and `base_map()` as in TP2a.

In [ ]:
import shutil
from datetime import datetime, timezone
from pathlib import Path

import copernicusmarine
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cmocean

# Study area: Pacific coast of Costa Rica (Gulf of Papagayo + Costa Rica Dome)
LON_MIN, LON_MAX = -92, -80
LAT_MIN, LAT_MAX = 4, 14

# Where your downloads go (created next to this notebook)
DATA_DIR = Path("data_tp2")
DATA_DIR.mkdir(exist_ok=True)

# Instructor's backup copy on Kabre, used only if a download fails
OBS_DIR = "/data/gcambon/COMMONDATA/DATASETS_PSF/"
if not Path(OBS_DIR).exists():          # not on Kabre: instructor's local copy
    OBS_DIR = "../data/DATASETS_PSF"
BACKUP_DIR = Path(OBS_DIR) / "TP2_BACKUP"


def download(fname, **kwargs):
    # Download a Copernicus Marine subset to DATA_DIR/fname, skip if it already exists
    todo = [k for k, v in kwargs.items() if v is ...]
    if todo:
        raise ValueError(f"Complete the TODOs first: {todo}")
    out = DATA_DIR / fname
    if out.exists():
        print(f"✓ {out} already there, download skipped ({out.stat().st_size / 1e6:.1f} MB)")
        return out
    try:
        copernicusmarine.subset(output_directory=str(DATA_DIR), output_filename=fname, **kwargs)
    except Exception as err:
        backup = BACKUP_DIR / fname
        if not backup.exists():
            raise
        print(f"⚠ Download failed ({err}).\n  Using the instructor's backup copy: {backup}")
        shutil.copy(backup, out)
    print(f"✓ {out} ({out.stat().st_size / 1e6:.1f} MB)")
    return out


def show_dataset(dataset_id):
    # Print variables, units and coordinate ranges of a Copernicus Marine dataset
    cat = copernicusmarine.describe(dataset_id=dataset_id, disable_progress_bar=True)
    product = cat.products[0]
    part = product.datasets[0].versions[0].parts[0]
    service = next(s for s in part.services if s.service_name == "arco-geo-series")
    print(f"{product.title}\n  product : {product.product_id}\n  dataset : {dataset_id}\n")
    for var in service.variables:
        print(f"  variable {var.short_name!r:20} units: {var.units:10} ({var.standard_name})")
    print()
    for coord in service.variables[0].coordinates:
        if coord.coordinate_id == "time":
            t0, t1 = (datetime.fromtimestamp(v / 1000, tz=timezone.utc).date()
                      for v in (coord.minimum_value, coord.maximum_value))
            print(f"  time      : {t0} → {t1}")
        elif coord.values:
            print(f"  {coord.coordinate_id:9} : {len(coord.values)} levels, {coord.values[0]} → {coord.values[-1]}")
        else:
            print(f"  {coord.coordinate_id:9} : {coord.minimum_value:.3f} → {coord.maximum_value:.3f}, step {coord.step:.3f}")


def base_map(ax, title="", extent=None):
    # extent = [lon_min, lon_max, lat_min, lat_max], the Costa Rica box by default
    ax.set_extent(extent or [LON_MIN, LON_MAX, LAT_MIN, LAT_MAX], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND.with_scale("50m"), facecolor="#d4c9a8", zorder=2)
    ax.add_feature(cfeature.COASTLINE.with_scale("50m"), linewidth=0.6, zorder=3)
    ax.add_feature(cfeature.BORDERS.with_scale("50m"), linewidth=0.4, linestyle="--", zorder=3)
    gl = ax.gridlines(draw_labels=True, linewidth=0.3, color="gray", alpha=0.5, linestyle="--")
    gl.top_labels = False
    gl.right_labels = False
    if title:
        ax.set_title(title, fontsize=11)
    return ax


print(f"Study area: lon [{LON_MIN}, {LON_MAX}]  lat [{LAT_MIN}, {LAT_MAX}]")
print(f"Downloads go to: {DATA_DIR.resolve()}")

---
## Exercise 1 — Download 31 years, the light way

31 years of **daily**, **3D** data over our box would weigh several GB. For a climatology of the **surface** temperature we only need:
- **monthly** means: the dataset `cmems_mod_glo_phy_my_0.083deg_P1M-m` (GLORYS12, the one of TP2a);
- **one depth level**: the first one, at 0.49 m (`minimum_depth=0, maximum_depth=1`);
- one variable: `thetao`.

**🎯 Your turn**: download `thetao` at the surface over the study box, from **January 1993 to December 2023**.

In [ ]:
### STUDENT: # TODO: complete the arguments
SST_FILE = download(
    "glorys12_sst_costarica_1993-2023.nc",
    dataset_id="cmems_mod_glo_phy_my_0.083deg_P1M-m",
    variables=["thetao"],
    minimum_longitude=LON_MIN, maximum_longitude=LON_MAX,
    minimum_latitude=LAT_MIN, maximum_latitude=LAT_MAX,
### STUDENT:     minimum_depth=...,     # TODO
### STUDENT:     maximum_depth=...,     # TODO
### STUDENT:     start_datetime=...,    # TODO
### STUDENT:     end_datetime=...,      # TODO
### BEGIN SOLUTION
    minimum_depth=0, maximum_depth=1,
    start_datetime="1993-01-01",
    end_datetime="2023-12-31",
### END SOLUTION
)

ds = xr.open_dataset(SST_FILE)
sst = ds["thetao"].isel(depth=0)          # drop the single depth level: (time, latitude, longitude)
print(ds)
### HINT: Surface only: `minimum_depth=0, maximum_depth=1` (the first model level is at 0.49 m).
### HINT: Dates: `"1993-01-01"` to `"2023-12-31"`.

**✅ Check**: a file of ~13 MB, `time: 372` (31 years × 12 months, each dated on the 1st), `depth: 1` (0.49 m), `latitude: 121`, `longitude: 145`. The temperature is already in °C (`degrees_C`): no conversion needed.

💬 You may see a `WARNING` about the depth range: harmless (as in TP2a).

---
## Exercise 2 — One number per month: the box mean

To follow the temperature through time, average the map over the box. On a longitude–latitude grid the cells get smaller towards the poles, so we weight each cell by **cos(latitude)** (the area-weighted mean of TP1b, advanced track B).

**🎯 Your turn**: compute the area-weighted mean of `sst` over latitude and longitude.

In [ ]:
weights = np.cos(np.deg2rad(sst.latitude))

### STUDENT: # TODO: area-weighted mean over latitude and longitude -> one value per month
### STUDENT: series = ...
### BEGIN SOLUTION
series = sst.weighted(weights).mean(["latitude", "longitude"])
### END SOLUTION

fig, ax = plt.subplots(figsize=(14, 4))
series.plot(ax=ax, linewidth=1)
ax.set_ylabel("SST (°C)")
ax.set_title("GLORYS12 — monthly SST averaged over the box, 1993–2023")
ax.grid(alpha=0.3)
plt.show()
print(f"{series.sizes['time']} months, from {float(series.min()):.2f} to {float(series.max()):.2f} °C, mean {float(series.mean()):.2f} °C")
### HINT: `sst.weighted(weights).mean(["latitude", "longitude"])`.

**✅ Check**: 372 values between **≈ 26.0 and ≈ 30.1 °C** (mean ≈ 28.1 °C). The curve wiggles every year (the seasons), and some years stand out as warmer (1997–98, 2015, 2023) or colder (1999–2000, 2010–11) as a whole. Separating these two kinds of variations is the aim of the next exercises.

---
## Exercise 3 — The climatology: the typical year

`series.groupby("time.month")` puts the 372 months into **12 groups**: all the Januaries, all the Februaries… `.mean()` averages each group. The result has a new dimension `month` (1 to 12): the **monthly climatology** over 1993–2023.

**🎯 Your turn**: compute the climatology and plot it.

In [ ]:
### STUDENT: # TODO: monthly climatology (12 values)
### STUDENT: clim = ...
### BEGIN SOLUTION
clim = series.groupby("time.month").mean()
### END SOLUTION

fig, ax = plt.subplots(figsize=(8, 4))
clim.plot(ax=ax, marker="o")
ax.set_xticks(range(1, 13))
ax.set_xticklabels(["J", "F", "M", "A", "M", "J", "J", "A", "S", "O", "N", "D"])
ax.set_ylabel("SST (°C)")
ax.set_title("Monthly climatology of the box SST, 1993–2023")
ax.grid(alpha=0.3)
plt.show()
print(clim.round(2).values)
print(f"Seasonal amplitude: {float(clim.max() - clim.min()):.2f} °C")
### HINT: `series.groupby("time.month").mean()`.

**✅ Check**: coldest in **January** (≈ 27.4 °C), warmest in **May** (≈ 28.9 °C): a seasonal amplitude of **≈ 1.4 °C**. The winter minimum is the Papagayo wind and upwelling season (TP2a).

---
## Exercise 4 — Anomalies: what was unusual?

Subtracting the climatology **month by month** removes the seasonal cycle: each January minus the mean January, each February minus the mean February… With xarray, one line: `series.groupby("time.month") - clim`.

**🎯 Your turn**: compute the anomalies. The cell then draws them in red (warmer than usual) and blue (colder), with the main El Niño and La Niña events shaded.

In [ ]:
### STUDENT: # TODO: anomalies = series minus the climatology of the same calendar month
### STUDENT: anom = ...
### BEGIN SOLUTION
anom = series.groupby("time.month") - clim
### END SOLUTION

events = {"El Niño": [("1997-05", "1998-05"), ("2015-03", "2016-05"), ("2023-05", "2024-01")],
          "La Niña": [("1998-07", "2000-12"), ("2010-06", "2011-05")]}

fig, ax = plt.subplots(figsize=(14, 4))
ax.fill_between(anom.time.values, anom.values, 0, where=anom.values > 0, color="#d6604d", label="warmer than usual")
ax.fill_between(anom.time.values, anom.values, 0, where=anom.values < 0, color="#4393c3", label="colder than usual")
for name, periods in events.items():
    for start, end in periods:
        ax.axvspan(np.datetime64(start), np.datetime64(end), alpha=0.12,
                   color="#b2182b" if name == "El Niño" else "#2166ac")
        ax.text(np.datetime64(start), 2.05 if name == "El Niño" else -1.9, name, fontsize=8)
ax.axhline(0, color="k", linewidth=0.8)
ax.set_ylim(-2, 2.3)
ax.set_ylabel("SST anomaly (°C)")
ax.set_title("Box SST anomaly (observation − climatology), 1993–2023")
ax.legend(loc="lower left")
ax.grid(alpha=0.3)
plt.show()

print("Warmest months:\n", anom.to_series().nlargest(5).round(2))
print("\nColdest months:\n", anom.to_series().nsmallest(5).round(2))
### HINT: `series.groupby("time.month") - clim`: xarray matches each month with its climatological value.

**✅ Check**:
- Anomalies between **≈ −1.4 and ≈ +1.9 °C**, standard deviation ≈ 0.5 °C; their mean over 1993–2023 is 0 (by construction).
- The warmest months: **December 2015 – January 2016** (+1.9 °C), **September–October 2023** (+1.8 °C), March 1998 (+1.35 °C): the three strong **El Niño** events. The coldest: December 2010 (−1.4 °C) and January 2000 (−1.35 °C): **La Niña**.

💬 **Discuss**:
- The seasonal amplitude is ~1.4 °C; a strong El Niño adds up to ~2 °C. Here, **interannual variability is as large as the seasonal cycle**.
- **2023**, the year of TP1b and TP2a, was **not a normal year**: an El Niño was developing, and it is the warmest year of the series (annual mean anomaly +1.1 °C). A single year is never a climatology (Course 1).

---
## Exercise 5 — Cross-check: the climatology without `groupby`

Is `groupby` really doing what we think? The 372 months are 31 complete years, in order: reshaped into a table of **31 rows (years) × 12 columns (months)**, the mean of each column is the climatology. Plain numpy, an independent method.

In [ ]:
### STUDENT: # TODO: reshape the 372 values into (31, 12), then average over the years (axis 0)
### STUDENT: clim_numpy = ...
### BEGIN SOLUTION
clim_numpy = series.values.reshape(31, 12).mean(axis=0)
### END SOLUTION

for m in range(12):
    print(f"month {m + 1:2d}:  groupby = {float(clim[m]):.4f}   numpy = {clim_numpy[m]:.4f}")
print(f"\nLargest difference: {np.abs(clim.values - clim_numpy).max():.1e} °C")
### HINT: `series.values` is a plain numpy array of 372 values: `.reshape(31, 12)` then `.mean(axis=0)`.

**✅ Check**: the two climatologies are identical (largest difference ~1e-14 °C, rounding noise). This works only because the series starts in January and contains complete years: with a series starting in March, the columns would not be calendar months — `groupby("time.month")` does not have this problem.

---
## 🐞 Debugging exercise

A colleague computed anomalies in their own way. Their cell has **two bugs**: one raises an error, the other is **silent** (it runs, but the result is wrong). Run it, read the error, fix it, then compare the plot with Exercise 4.

In [ ]:
### STUDENT: # 🐞 Fix me! (2 bugs)
### STUDENT: clim_bug = series.groupby("month").mean()
### STUDENT: anom_bug = series - series.mean()
### BEGIN SOLUTION
# Fixed version
clim_bug = series.groupby("time.month").mean()
anom_bug = series.groupby("time.month") - clim_bug
### END SOLUTION

fig, ax = plt.subplots(figsize=(14, 3.5))
anom_bug.plot(ax=ax, linewidth=1)
ax.axhline(0, color="k", linewidth=0.8)
ax.set_ylabel("SST anomaly (°C)")
ax.grid(alpha=0.3)
plt.show()
print(f"Standard deviation of the anomalies: {float(anom_bug.std()):.2f} °C")
### BEGIN SOLUTION

# Bug 1: KeyError: 'month'. The month is not a variable of the series: it is a property
#        of the time coordinate, written "time.month".
# Bug 2 (silent): series - series.mean() removes ONE number (the 31-year mean), not the
#        climatology of each month: the seasonal cycle is still there. The curve wiggles
#        every year and the standard deviation is 0.70 °C instead of 0.53 °C.
### END SOLUTION
### NO REVEAL
### HINT: Bug 1: read the last line of the error. Where does the month come from? (Exercise 3.)
### HINT: Bug 2: is `clim_bug` used anywhere? Compare the curve with the one of Exercise 4: does it still wiggle every year?

**✅ Check**: the fixed cell gives the same curve as Exercise 4, with a standard deviation of **0.53 °C**. With the silent bug, the curve still shows a regular yearly wiggle and the standard deviation is 0.70 °C: the seasonal cycle has not been removed.

---
### 🚀 Advanced track A — anomaly maps: El Niño vs La Niña

The same `groupby` works on the **maps**: `sst.groupby("time.month") - sst.groupby("time.month").mean()` gives an anomaly map for every month. Map **October 2023** (El Niño) and **December 2010** (La Niña) side by side, with the same **diverging** colour scale (`cmap="RdBu_r"`, `vmin=-3`, `vmax=3`).

Is the anomaly uniform over the box? Where is it strongest?

In [ ]:
### STUDENT: # 🚀 Your code here
### BEGIN SOLUTION
anom_map = sst.groupby("time.month") - sst.groupby("time.month").mean()

fig, axes = plt.subplots(1, 2, figsize=(16, 5.5), subplot_kw={"projection": ccrs.PlateCarree()})
for ax, date, title in zip(axes, ["2023-10", "2010-12"], ["October 2023 (El Niño)", "December 2010 (La Niña)"]):
    im = anom_map.sel(time=date).squeeze().plot(ax=ax, transform=ccrs.PlateCarree(), cmap="RdBu_r",
                                                vmin=-3, vmax=3, add_colorbar=False)
    base_map(ax, f"SST anomaly, {title}")
fig.colorbar(im, ax=axes, shrink=0.8, label="SST anomaly (°C)")
plt.show()
### END SOLUTION
### HINT: `anom_map.sel(time="2023-10").squeeze()` gives one map; use `add_colorbar=False` and one shared `fig.colorbar`.

**✅ Check (advanced)**: October 2023 is warmer than usual **everywhere** in the box (+0.8 to +2.9 °C, +1.8 on average); December 2010 colder almost everywhere (down to −2.9 °C, −1.4 on average). In both cases the anomaly is **not uniform**: it is strongest in the open Pacific south-west of Costa Rica (≈ 7–9°N, 86–88°W), weaker on the Caribbean side. In December 2010 the north-west of the box is close to normal: the large-scale signal is mixed with local events.

### 🚀 Advanced track B — the Gulf of Papagayo alone

Repeat Exercises 2–4 for a small box around the Gulf of Papagayo only (89°W–86°W, 9.5°N–11.5°N). Are the anomalies larger or smaller than for the whole box? Why?

In [ ]:
### STUDENT: # 🚀 Your code here
### BEGIN SOLUTION
pap = sst.sel(longitude=slice(-89, -86), latitude=slice(9.5, 11.5))
pap_series = pap.weighted(np.cos(np.deg2rad(pap.latitude))).mean(["latitude", "longitude"])
pap_anom = pap_series.groupby("time.month") - pap_series.groupby("time.month").mean()

fig, ax = plt.subplots(figsize=(14, 4))
anom.plot(ax=ax, label="whole box", linewidth=1)
pap_anom.plot(ax=ax, label="Gulf of Papagayo", linewidth=1)
ax.axhline(0, color="k", linewidth=0.8)
ax.set_ylabel("SST anomaly (°C)")
ax.legend()
ax.grid(alpha=0.3)
plt.show()
print(f"std: whole box {float(anom.std()):.2f} °C, Papagayo {float(pap_anom.std()):.2f} °C")
print("Papagayo, warmest months:\n", pap_anom.to_series().nlargest(3).round(2))
### END SOLUTION
### HINT: `sst.sel(longitude=slice(-89, -86), latitude=slice(9.5, 11.5))`, then the same three steps as Exercises 2, 3 and 4.

**✅ Check (advanced)**: the Papagayo anomalies are larger (standard deviation ≈ 0.75 °C against 0.53 °C), up to **+2.8 °C in February 1998** and +2.5 °C in December 2015. A small box averages out less: local events (a weaker or stronger wind jet) add to the large-scale El Niño signal.

---
## Recap

| Step | Code |
|---|---|
| area-weighted box mean | `sst.weighted(np.cos(np.deg2rad(sst.latitude))).mean(["latitude", "longitude"])` |
| monthly climatology | `series.groupby("time.month").mean()` |
| anomalies | `series.groupby("time.month") - clim` |
| one month | `anom.sel(time="2023-10")` |

- The **climatology** is the typical year (here 1993–2023); the **anomaly** is what was unusual.
- In this region, **El Niño and La Niña** change the SST by up to ~2 °C, as much as the seasons.
- **2023 was an El Niño year**: keep it in mind when you compare it with a model or a climatology on Thursday.

Your file `data_tp2/glorys12_sst_costarica_1993-2023.nc` stays available for the mini-project.

<!-- SOLUTION ONLY -->
### 👩‍🏫 Instructor notes

- **Before the workshop**: put a copy of `glorys12_sst_costarica_1993-2023.nc` in `BACKUP_DIR` on Kabre.
- **Timing** (~1h30): Exercises 1–2 ≈ 25 min, Exercises 3–4 ≈ 35 min, cross-check ≈ 10 min, debugging ≈ 10 min.
- **Climatology period**: the WMO standard is 30 years (e.g. 1991–2020). Here we use all available years, 1993–2023, for simplicity; mention that including 2023 slightly warms the climatology.
- **ENSO dates**: the shaded periods are approximate, to guide the eye. For an official reference, show NOAA's Oceanic Niño Index (ONI) table.
- **GLORYS12 is a reanalysis** (model + data assimilation), not an observation: for SST it is strongly constrained by satellite data, so the anomalies are reliable; for a pure-observation climatology, OSTIA or OISST would do, but they are heavier to download (daily).